# Algebraic numbers on a Colab GPU

1. Runtime → Change runtime type → Hardware accelerator → **T4 GPU** (free) or **A100** (paid).
2. Runtime → Run all.

The first cell clones this repo if `algebraics/` is missing. GPU work is batched **float64** companion-matrix eigenvalues. That is the highest precision CUDA gives you.

What adds detail is a larger `MAXH`. Brooks used 15. A free T4 is enough for 17. An A100 can do 18.

In [ ]:
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("algebraics/algebraics.py").exists():
    !git clone --depth 1 https://github.com/ST-48-1240162/constellatio-numerorum.git /content/constellatio-numerorum
    os.chdir("/content/constellatio-numerorum")

import torch
print("cwd", Path.cwd())
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), end=" ")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("(enable a GPU runtime or this falls back to CPU)")

In [ ]:
MAXH = 17
WIDTH = 7680
HEIGHT = 4320
BATCH = 8192
# zoom defaults to height/5 (=864); same framing as 3840×2160, 2× px density

from algebraics.algebraics_gpu import load_or_compute_gpu, pick_device, render_png

device = pick_device()
points = load_or_compute_gpu(MAXH, device=device, batch=BATCH)
print("roots", len(points))

out = Path("algebraics") / f"algebraics_h{MAXH}_{WIDTH}px.png"
render_png(points, out, width=WIDTH, height=HEIGHT)
print("wrote", out)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

png = Path("algebraics") / f"algebraics_h{MAXH}_{WIDTH}px.png"
display(Image(filename=str(png), width=960))

if IN_COLAB:
    from google.colab import files
    files.download(str(png))

## Single-colour degree plots

One full-map PNG/PDF per hue-table entry (**deg1** … **deg8**, **deg9_plus**), same **7680×4320** viewport as above. Each plot shows exactly one degree (or ≥9 for white). Also writes `points.npz` and `summary.txt`.

Needs `pdflatex` (installed below on Colab).

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y texlive-latex-base texlive-fonts-recommended > /dev/null
    !pip -q install -r requirements.txt

from algebraics.algebraics import consolidate
from algebraics.regions.build import build_degree_plots

REGION_WIDTH, REGION_HEIGHT = WIDTH, HEIGHT
blobs, weights = consolidate(points)
build_degree_plots(blobs, weights, maxh=MAXH, width=REGION_WIDTH, height=REGION_HEIGHT)
print(f"wrote algebraics/regions/deg*  ({REGION_WIDTH}x{REGION_HEIGHT})")

## Constellatio annotated PDF

Overlay callout labels on the full image and compile `constellatio.pdf`.

In [ ]:
from algebraics.annotate_constellatio import build_pdf

build_pdf(out)

In [ ]:
import shutil
from IPython.display import Image, display

for pid in ("deg1", "deg2", "deg3", "deg4", "deg5", "deg6", "deg7", "deg8", "deg9_plus"):
    p = Path("algebraics/regions") / pid / f"{pid}.png"
    if p.exists():
        print(pid)
        display(Image(filename=str(p), width=480))

if IN_COLAB:
    zip_path = Path("/content/constellatio_regions.zip")
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", "algebraics/regions")
    from google.colab import files
    files.download(str(zip_path))
    files.download("algebraics/constellatio.pdf")